# Solution: Router Chains (Runnables) + `@chain` Decorator (Use-Case Based)

This notebook contains **complete solutions** for:
- Warm-up `normalize_query` using `@chain`
- Use Case 1: Customer Support Triage Router
- Use Case 2: Data Assistant vs Coding Assistant Router
- Use Case 3: Sentiment Routing for Response Style
- Bonus: Confidence Gate wrapper

> Notes:
> - This solution uses **deterministic routing rules** for the routers.
> - Model calls are used to generate responses (LLM) but routing is **not** decided by the model.
> - Do not paste API keys in the notebook; use environment variables.


In [3]:
from __future__ import annotations

import json
import re
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.runnables import chain

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature =0)
parser= StrOutputParser()

## 1) Warm-up — `normalize_query` (`@chain`)

In [4]:
@chain
def normalize_query(x: str) -> str:
    """Normalize user query: strip + collapse spaces+lowercase."""

    x= x.strip().lower()
    x=re.sub(r"\s+", " ", x)
    return x


print(normalize_query.invoke("   Refund status for Order #1234"))

refund status for order #1234


## 2) Use Case 1 — Customer Support Triage Router (Refund / Tech / General)

We build three specialized chains and a deterministic router.


In [10]:
#Prompts
refund_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a customer support agent. Follow the refund policy:"
     "1) Apologize. 2) Ask for order id if missing. 3) If item is damaged, mention replacement/refund option" 
     "Be concise, professional"),
     ("human", "{ticket}")  
])

ṭech_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a technical support agent. Provide step-by-step troubleshooting"
     "Ask for a device/OS/app version only if missing. Be concise"),
     ("human","{ticket}")
])

general_prompt = ChatPromptTemplate.from_messages([
    ("system",
    "You are a customer support agent for general inquiries (delivery, store hours, membership)."
    "If the request is unclear, ask ONE clarifying question. Be friendly and concise."),
    ("human","{ticket}")
    ])

@chain
def refund_chain(ticket: str) -> str:
    return (refund_prompt | llm | parser).invoke({"ticket": ticket})

@chain
def tech_chain(ticket: str) -> str:
    return (tech_prompt | llm | parser).invoke({"ticket": ticket})

@chain
def general_chain(ticket:str) -> str:
    return (general_prompt | llm | parser).invoke({"ticket": ticket})


def route_support(ticket: str) -> Literal["refund", "tech", "general"]:
    t = normalize_query.invoke(ticket)

    refund_kw =["refund", "money back", "return", "cancel", "damaged", "broken", "defect", "replacement"]
    tech_kw = ["app", "crash", "login", "error", "bug", "wifi", "internet", "payment failed", "otp", "issue", "not working"]


    if any(k in t for k in refund_kw):
        return "refund"
    if any(k in t for k in tech_kw):
        return "tech"
    return "general"

support_router = RunnableBranch(
    (lambda x: route_support(x)== "refund", refund_chain),
    (lambda x: route_support(x)== "tech", tech_chain),
    general_chain
)

tests = [ "I want my money back. My order arrived damaged.",
    "App is crashing after login. I'm on iPhone 14.",
    "Do you deliver on Sundays?",
    "Help",
]


for t in tests:
    print("\n -- \n Ticket", t)
    print("Route:", route_support(t))
    print(support_router.invoke(t))


 -- 
 Ticket I want my money back. My order arrived damaged.
Route: refund
I apologize for the inconvenience you've experienced. Could you please provide your order ID so I can assist you further? If the item is damaged, we can offer you a replacement or a refund.

 -- 
 Ticket App is crashing after login. I'm on iPhone 14.
Route: tech


NameError: name 'tech_prompt' is not defined

## 3) Use Case 2 — Data Assistant vs Coding Assistant Router

Deterministic keyword routing; two `@chain` assistants with formatting constraints.


In [11]:
data_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data assistant. Explain clearly in simple terms. Provide ONE Small example(no code unless absolutely required)"
     "End your response with exaclty : 'next step: <oneaction>'"),
     ("human", "{q}")
])

code_prompt = ChatPromptTemplate.from_messages([
    ("system",
    "you are a coding assistant. Provide a correct Python solution."
    "Include : 1)short explanation 2) a Python code block 3) edge cases 4) complexity note."
    "Be concise but complete."),
    ("human", "{q}")
])

@chain
def data_assistant(q:str)-> str:
    return (data_prompt | llm | parser).invoke({"q":q})

@chain
def coding_assistant(q:str)-> str:
    return (code_prompt | llm| parser).invoke({"q":q})

def route_mode(q:str) -> Literal["data", "code"]:
    t = normalize_query.invoke(q)
    code_kw=["code","python","pandas","sql","implement","function","bug","error","stack trace","write","script"]
    if any(k in t for k in code_kw):
        return "code"

    return "data"

assistant_router = RunnableBranch(
    (lambda x: route_mode(x)== "code", coding_assistant),
    data_assistant,
)
tests = [
    "Explain precision vs recall with an example",
    "Write python code to remove duplicates from a list but keep order",
    "Bug: my pandas merge is creating duplicates, what should I do?",
]
for t in tests:
    print("\n---\nQuery:", t)
    print("Route:", route_mode(t))
    print(assistant_router.invoke(t))



---
Query: Explain precision vs recall with an example
Route: data
Precision and recall are two important metrics used to evaluate the performance of a classification model, especially in situations where the classes are imbalanced.

- **Precision** measures how many of the items that the model predicted as positive are actually positive. In simple terms, it answers the question: "Of all the items I labeled as positive, how many were correct?"

- **Recall** measures how many of the actual positive items were correctly identified by the model. It answers the question: "Of all the actual positive items, how many did I find?"

**Example:**
Imagine you have a basket of 100 fruits, where 20 are apples (the positive class) and 80 are oranges (the negative class). 

If your model predicts that 30 fruits are apples, but only 15 of those are actually apples (the rest are oranges), then:
- **Precision** = 15 (correctly predicted apples) / 30 (total predicted apples) = 0.5 or 50%
- **Recall** = 

## 4) Use Case 3 — Sentiment Routing for Response Style

We'll use simple deterministic sentiment rules for the assignment.


In [14]:
def classify_sentiment(text:str) -> Literal["positive","negative","neutral"]:
    t = normalize_query.invoke(text)

    pos_kw = ["amazing", "great", "excellent", "love", "helpful","fantastic", "awesome", "super", "thanks",  "good"]
    neg_kw = ["bad", "terrible","too fast", "confusing", "lost", "hate", "waste", "boring", "not clear","worst", "slow"]
    if any(k in t for k in pos_kw):
        return "positive"
    if any(k in t for k in neg_kw):
        return "negative"

    return "neutral"
positive_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a course feedback assistant. Reply warmly. Thank them and ask for permission to use their feedback as a testimonial. "
     "Keep it under 6 lines."),
    ("human", "{fb}")
])

negative_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a course feedback assistant. Apologize and take accountability. Ask exactly TWO clarifying questions. "
     "Offer to help with a practical next step. Keep it under 8 lines."),
    ("human", "{fb}")
])

neutral_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a course feedback assistant. Acknowledge neutrally. Ask what can be improved or what they want next. "
     "Keep it under 6 lines."),
    ("human", "{fb}")
])

@chain
def positive_reply(fb: str) -> str:
    return (positive_prompt | llm | parser).invoke({"fb": fb})

@chain
def negative_reply(fb: str) -> str:
    return (negative_prompt | llm | parser).invoke({"fb": fb})

@chain
def neutral_reply(fb: str) -> str:
    return (neutral_prompt | llm | parser).invoke({"fb": fb})

feedback_router = RunnableBranch(
    (lambda x: classify_sentiment(x) == "positive", positive_reply),
    (lambda x: classify_sentiment(x) == "negative", negative_reply),
    neutral_reply,  # includes fallback behavior by design
)

tests = [
    "This was amazing, I finally understand RunnableBranch!",
    "The pace was too fast and I got lost.",
    "The session was okay.",
    "???",
]
for t in tests:
    print("\n---\nFeedback:", t)
    print("Sentiment:", classify_sentiment(t))
    print(feedback_router.invoke(t))






---
Feedback: This was amazing, I finally understand RunnableBranch!
Sentiment: positive
Thank you so much for your kind words! We're thrilled to hear that you found the course helpful. Would you mind if we used your feedback as a testimonial? It would really help others who are considering the course.

---
Feedback: The pace was too fast and I got lost.
Sentiment: negative
I’m really sorry to hear that the pace of the course was too fast for you. I take full accountability for that experience. Could you share which specific topics you found most challenging? Also, were there particular sections where you felt the pace was overwhelming? 

As a next step, I can help you find additional resources or schedule a review session to go over the material at a more comfortable pace.

---
Feedback: The session was okay.
Sentiment: neutral
Thank you for your feedback. What aspects do you think could be improved? Is there anything specific you would like to see in future sessions?

---
Feedback: 

In [15]:
confidence_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict classifier. Return ONLY valid JSON with keys: confidence, reason. "
     "confidence must be one of: high, medium, low. "
     "Assess whether the answer is likely correct and sufficiently specific given the input."),
    ("human",
     "User input: {user_input}\n\nDraft answer: {answer}\n\nReturn JSON only.")
])

@chain
def classify_confidence(inp: dict) -> str:
    return (confidence_prompt | llm | parser).invoke(inp)

def gate_result(d: dict) -> str:
    """If low confidence, ask clarifying question. Otherwise return answer."""
    raw = d.get("confidence_json", "")
    answer = d.get("answer", "")
    try:
        obj = json.loads(raw)
        conf = obj.get("confidence", "medium")
    except Exception:
        conf = "medium"

    if conf == "low":
        return (
            "Sorry — I may not have enough details to answer that reliably. "
            "Could you share a bit more context (e.g., order id / exact error message / what you tried so far)?"
        )
    return answer

# Build a gated pipeline:
# 1) produce answer via support_router
# 2) attach confidence JSON
# 3) gate

def support_answer(ticket: str) -> dict:
    return {"user_input": ticket, "answer": support_router.invoke(ticket)}

gated_support_router = (
    RunnableLambda(support_answer)
    | RunnableLambda(lambda d: {**d, "confidence_json": classify_confidence.invoke(d)})
    | RunnableLambda(gate_result)
)

# Test with ambiguous and clear inputs
tests = [
    "Help me with it",
    "I want a refund for my damaged order",
]
for t in tests:
    print("\n---\nTicket:", t)
    print(gated_support_router.invoke(t))



---
Ticket: Help me with it
Sure! Could you please clarify what specific assistance you need?

---
Ticket: I want a refund for my damaged order
I apologize for the inconvenience. Could you please provide your order ID so I can assist you further? If the item is damaged, we can offer you a replacement or a refund.
